In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# MY CODE 
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from scipy import stats
from scipy.special import boxcox1p
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("HOUSE PRICE PREDICTION - TRAIN & TEST PIPELINE")
print("="*70)

# ============================================================================
# STEP 1: LOAD TRAIN AND TEST DATA
# ============================================================================
print("\n" + "="*70)
print("STEP 1: LOADING DATA")
print("="*70)

train = pd.read_csv(r"/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv(r"/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print(f"✓ Train shape: {train.shape}")
print(f"✓ Test shape: {test.shape}")

# Save test IDs for submission
test_ids = test['Id'].copy()

# ============================================================================
# STEP 2: COMBINE TRAIN AND TEST FOR CONSISTENT PREPROCESSING
# ============================================================================
print("\n" + "="*70)
print("STEP 2: COMBINING TRAIN & TEST FOR PREPROCESSING")
print("="*70)

# Separate target from train
y_train = train['SalePrice'].copy()
train = train.drop(columns=['SalePrice'])

# Mark which rows are train/test
train['is_train'] = 1
test['is_train'] = 0

# Combine
df_all = pd.concat([train, test], axis=0, ignore_index=True)
print(f"✓ Combined shape: {df_all.shape}")

# ============================================================================
# STEP 3: ADVANCED FEATURE ENGINEERING
# ============================================================================
print("\n" + "="*70)
print("STEP 3: FEATURE ENGINEERING")
print("="*70)

def create_features(df):
    """Create powerful derived features"""
    df = df.copy()
    
    # Total SF
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    
    # Total Bathrooms
    df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath'] + 
                       df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
    
    # Total Porch SF
    df['TotalPorchSF'] = (df['OpenPorchSF'] + df['3SsnPorch'] + 
                          df['EnclosedPorch'] + df['ScreenPorch'] + 
                          df['WoodDeckSF'])
    
    # Age features
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    
    # Quality interactions
    df['OverallScore'] = df['OverallQual'] * df['OverallCond']
    df['QualitySF'] = df['GrLivArea'] * df['OverallQual']
    
    # Boolean features
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    
    # SF per Room
    df['SF_per_Room'] = df['GrLivArea'] / (df['TotRmsAbvGrd'] + 1)
    
    return df

df_all = create_features(df_all)
print(f"✓ Features created. New shape: {df_all.shape}")

# ============================================================================
# STEP 4: REMOVE UNNECESSARY COLUMNS
# ============================================================================
print("\n" + "="*70)
print("STEP 4: REMOVING UNNECESSARY COLUMNS")
print("="*70)

columns_to_remove = [
    'Id', 'Street', 'Utilities', 'Condition2', 'RoofMatl', 'Heating',
    'PoolQC', 'MiscFeature', 'Alley', 'GarageYrBlt', 'LandContour',
    'LotConfig', 'LandSlope', 'BsmtFinType2', 'Electrical', 'Functional',
    'GarageCond', 'MiscVal', 'MoSold'
]

columns_to_remove = [col for col in columns_to_remove if col in df_all.columns]
df_all = df_all.drop(columns=columns_to_remove)

print(f"✓ Removed {len(columns_to_remove)} columns")

# ============================================================================
# STEP 5: IDENTIFY NUMERICAL AND CATEGORICAL COLUMNS
# ============================================================================
numerical_cols = df_all.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('is_train')  # Don't treat as feature
categorical_cols = df_all.select_dtypes(include=['object']).columns.tolist()

print(f"\n✓ Numerical columns: {len(numerical_cols)}")
print(f"✓ Categorical columns: {len(categorical_cols)}")

# ============================================================================
# STEP 6: HANDLE MISSING VALUES
# ============================================================================
print("\n" + "="*70)
print("STEP 5: HANDLING MISSING VALUES")
print("="*70)

missing_before = df_all.isnull().sum().sum()

# Fill numerical with median
for col in numerical_cols:
    if df_all[col].isnull().sum() > 0:
        df_all[col].fillna(df_all[col].median(), inplace=True)

# Fill categorical with 'Missing'
for col in categorical_cols:
    if df_all[col].isnull().sum() > 0:
        df_all[col].fillna('Missing', inplace=True)

missing_after = df_all.isnull().sum().sum()
print(f"✓ Missing values: {missing_before} → {missing_after}")

# ============================================================================
# STEP 7: HANDLE SKEWNESS WITH BOX-COX TRANSFORM
# ============================================================================
print("\n" + "="*70)
print("STEP 6: CORRECTING SKEWNESS")
print("="*70)

skewed_features = df_all[numerical_cols].apply(lambda x: stats.skew(x)).sort_values(ascending=False)
high_skew = skewed_features[abs(skewed_features) > 0.5]

print(f"Found {len(high_skew)} highly skewed features")

for feature in high_skew.index:
    df_all[feature] = boxcox1p(df_all[feature], 0.15)

print(f"✓ Applied Box-Cox transform")

# ============================================================================
# STEP 8: ONE-HOT ENCODE CATEGORICAL VARIABLES
# ============================================================================
print("\n" + "="*70)
print("STEP 7: ONE-HOT ENCODING")
print("="*70)

df_all_encoded = pd.get_dummies(df_all, columns=categorical_cols, drop_first=True, dtype=int)
print(f"✓ After encoding: {df_all_encoded.shape}")

# ============================================================================
# STEP 9: SPLIT BACK INTO TRAIN AND TEST
# ============================================================================
print("\n" + "="*70)
print("STEP 8: SPLITTING BACK TO TRAIN & TEST")
print("="*70)

train_processed = df_all_encoded[df_all_encoded['is_train'] == 1].drop(columns=['is_train'])
test_processed = df_all_encoded[df_all_encoded['is_train'] == 0].drop(columns=['is_train'])

print(f"✓ Train processed: {train_processed.shape}")
print(f"✓ Test processed: {test_processed.shape}")

# ============================================================================
# STEP 10: HANDLE OUTLIERS IN TRAIN TARGET
# ============================================================================
print("\n" + "="*70)
print("STEP 9: REMOVING OUTLIERS FROM TRAINING DATA")
print("="*70)

z_scores = np.abs(stats.zscore(y_train))
outliers = z_scores > 3
print(f"Found {outliers.sum()} outliers")

train_processed = train_processed[~outliers]
y_train = y_train[~outliers]

print(f"✓ Train after outlier removal: {train_processed.shape}")

# ============================================================================
# STEP 11: LOG TRANSFORM TARGET (IMPORTANT FOR RMSE!)
# ============================================================================
print("\n" + "="*70)
print("STEP 10: LOG TRANSFORMING TARGET")
print("="*70)

y_train_log = np.log1p(y_train)
print(f"✓ Applied log1p transform to target")

# ============================================================================
# STEP 12: SCALE NUMERICAL FEATURES
# ============================================================================
print("\n" + "="*70)
print("STEP 11: SCALING FEATURES (ROBUST SCALER)")
print("="*70)

scaler = RobustScaler()

# Fit on train, transform both train and test
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_processed),
    columns=train_processed.columns
)

test_scaled = pd.DataFrame(
    scaler.transform(test_processed),
    columns=test_processed.columns
)

print(f"✓ Scaled train: {train_scaled.shape}")
print(f"✓ Scaled test: {test_scaled.shape}")

# ============================================================================
# STEP 13: TRAIN MULTIPLE MODELS
# ============================================================================
print("\n" + "="*70)
print("STEP 12: TRAINING MODELS")
print("="*70)

models = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ),
    'Ridge': Ridge(alpha=20),
    'Lasso': Lasso(alpha=0.0005, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)
}

cv_results = {}

for name, model in models.items():
    print(f"\n{'-'*70}")
    print(f"Training: {name}")
    
    # Cross-validation on log-transformed target
    cv_scores = cross_val_score(
        model, train_scaled, y_train_log,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    
    cv_rmse = -cv_scores.mean()
    cv_std = cv_scores.std()
    
    cv_results[name] = {'CV RMSE (log)': cv_rmse, 'CV Std': cv_std}
    
    print(f"  CV RMSE (log): {cv_rmse:.5f} (±{cv_std:.5f})")

# ============================================================================
# STEP 14: STACKING ENSEMBLE (BEST PERFORMANCE!)
# ============================================================================
print("\n" + "="*70)
print("STEP 13: CREATING STACKING ENSEMBLE")
print("="*70)

estimators = [
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)),
    ('ridge', Ridge(alpha=20))
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=10),
    cv=5,
    n_jobs=-1
)

print("Training Stacking Ensemble...")
stacking_model.fit(train_scaled, y_train_log)

# Cross-validation for stacking
cv_scores = cross_val_score(
    stacking_model, train_scaled, y_train_log,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

stacking_cv_rmse = -cv_scores.mean()
print(f"✓ Stacking CV RMSE (log): {stacking_cv_rmse:.5f}")

cv_results['Stacking Ensemble'] = {'CV RMSE (log)': stacking_cv_rmse, 'CV Std': cv_scores.std()}

# ============================================================================
# STEP 15: DISPLAY ALL MODEL RESULTS
# ============================================================================
print("\n" + "="*70)
print("MODEL COMPARISON (CROSS-VALIDATION)")
print("="*70)

results_df = pd.DataFrame(cv_results).T
results_df = results_df.sort_values('CV RMSE (log)')
print("\n" + results_df.to_string())

best_model_name = results_df['CV RMSE (log)'].idxmin()
print(f"\n🏆 BEST MODEL: {best_model_name}")

# ============================================================================
# STEP 16: MAKE PREDICTIONS ON TEST SET
# ============================================================================
print("\n" + "="*70)
print("STEP 14: MAKING PREDICTIONS ON TEST SET")
print("="*70)

# Use stacking ensemble for final predictions
print("Using Stacking Ensemble for predictions...")

# Predict (log scale)
predictions_log = stacking_model.predict(test_scaled)

# Transform back to original scale
predictions = np.expm1(predictions_log)

print(f"✓ Predictions made: {len(predictions)}")
print(f"  Min: ${predictions.min():,.2f}")
print(f"  Max: ${predictions.max():,.2f}")
print(f"  Mean: ${predictions.mean():,.2f}")

# ============================================================================
# STEP 17: CREATE SUBMISSION FILE
# ============================================================================
print("\n" + "="*70)
print("STEP 15: CREATING SUBMISSION FILE")
print("="*70)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

# Save to CSV
submission.to_csv('submission.csv', index=False)

print(f"✓ Submission file created: submission.csv")
print(f"  Shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10).to_string(index=False))

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*70)
print(f"Best Model: {best_model_name}")
print(f"CV RMSE (log): {results_df.loc[best_model_name, 'CV RMSE (log)']:.5f}")
print(f"\n📁 Submission file ready: submission.csv")
print(f"📊 Format: Id, SalePrice")
print("="*70)